# Burn cost demo: validate and save a version to SQL

Load the recipe from 02 and the rows from 01. Fit and validate, inspect the results, then save the version to local SQL Server.
Saving makes the version available for review. Notebook 06 selects the champion.


In [ ]:
DATABASE_MODE = "remote"  # "local" or "remote"
RUNTIME_MODULE = "demo_sql_runtime"  # e.g. "project_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = "PricingNotebookDemo"
ALLOW_REMOTE_WRITES = True  # Only the local demo database.

RECIPE_PATH = "prototype.toml"

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "pricing_models").is_dir()
)

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pricing_pipeline.notebook import (
    PricingDataset,
    ModelRecipe,
    fit_model,
    connect,
    save_model_version,
    register_model,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/burn_cost_demo"
DATASET_PATH = MODEL_DIR / ".local" / "dataset.joblib"

## Load the dataset


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)

In [ ]:
# Data.
dataset = PricingDataset.load(DATASET_PATH)
df = dataset.df
display({"Rows": len(df), "Columns": len(df.columns)})

## Load the model recipe

Groups, special levels and spline settings come from `prototype.toml`. No second feature dictionary is needed.
This creates an unfitted model. The weekly runner will instead read the champion's saved definition and fitted state from SQL.


In [ ]:
MODEL, raw_superglm_model = ModelRecipe.load(MODEL_DIR / RECIPE_PATH).build(dataset=dataset)

## Register the model

This records the demo model identity in SQL.

In [ ]:
model = register_model(pricing, MODEL, source_root=MODEL_DIR)

## Fit and validate

`fit_model` fits the model on each validation split, refits the final
model on all rows, and exports its rating tables and fitted model. It also
writes the dataset manifest and split evidence to the chosen database.
Review the returned metrics before saving the version.

Pass a scikit-learn or custom splitter as `MODEL.validation` when you need
grouped or time-based CV. Set `groups_column` for grouped splits. Time-series
splitters use the saved dataframe order. Use column-based validation when the
dataset already contains split assignments.


In [ ]:
# Fit and validate.
raw_candidate = fit_model(
    pricing,
    model=model,
    frame=df,
    superglm_model=raw_superglm_model,
    model_kind="RAW",
)
raw_candidate.metrics

## Optional: export the raw recipe

Uncomment the call to save the exact configuration used for this fit. Existing
files require replace=True. SQL assigns revisions when a build is saved.


In [ ]:
# raw_candidate.recipe.save(MODEL_DIR / "raw_model.toml")


## Save this version

Run the next cell after reviewing the metrics. It saves the rating package, model recipe and fitted monitoring baseline to SQL.
The package compresses the recipe automatically. There is no compression setting to add here.
Saving does not select the champion. That happens in notebook 06.


In [ ]:
raw_published = save_model_version(pricing, raw_candidate)
display(
    {
        "Model": raw_published.model_name,
        "Kind": raw_published.model_kind,
        "Package": raw_published.package_version,
        "Model version": raw_published.recipe_revision,
        "Recipe status": raw_published.recipe_status,
        "Manifest": raw_published.manifest_id,
        "State": raw_published.package_status,
        "Reused equivalent": raw_published.deduplicated,
    }
)

Next: **06_model_deployment.ipynb** reviews and promotes this package. Skip optional 04 and 05 for the first run. **08_inspect_sql.ipynb** shows what was stored.